# Qdrant Examples: Movie Database

This notebook demonstrates practical Qdrant features using a **movie database** as the running example.
Movies have genres, years, and ratings — perfect for showing filters, grouping, facets, and recommendations.

### What we cover
| # | Topic | In one sentence |
|---|-------|-----------------|
| 1 | Setup | Connect and load 15 movies |
| 2 | Filtered Search | Find similar items that also match conditions |
| 3 | Payload Management | Update, delete, and index metadata |
| 4 | Scroll / Pagination | Page through all stored points |
| 5 | Recommendation API | Find more like what you already like |
| 6 | Batch Operations | Many inserts or searches in one call |
| 7 | Grouped Search | Return at most N results per group |
| 8 | Count & Facets | Count matches and get category breakdowns |
| 9 | Cleanup | Delete the collection |


## 1. Setup & Connection

Load API credentials from `.env` and connect to Qdrant Cloud.
Then create a fresh **8-dimensional** collection called `movies`.

> **Vector dimensions explained:** Each of the 8 numbers represents how strongly
> a movie belongs to a conceptual axis — action, drama, comedy, sci-fi (2 dims each).


In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, MatchAny, Range,
    PayloadSchemaType,
    RecommendQuery, RecommendInput,
    QueryRequest,
)
from dotenv import load_dotenv
import os

load_dotenv()
client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))
print("Connected to Qdrant:", client.get_collections())

/Users/chaitanya/Development/AI/qdrant-rag/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Connected to Qdrant: collections=[CollectionDescription(name='hello_world')]


In [2]:
COLLECTION = "movies"

# Drop if exists, then create fresh
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f"Deleted existing '{COLLECTION}' collection")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=8, distance=Distance.COSINE),
)
print(f"Created collection '{COLLECTION}' with 8-dim cosine vectors")

Created collection 'movies' with 8-dim cosine vectors


In [ ]:
# fmt: off
# 8-dim vectors: [action1, action2, drama1, drama2, comedy1, comedy2, scifi1, scifi2]
# Values closer to 1.0 on a pair of dims = strong genre signal
MOVIES = [
    # --- Action ---
    PointStruct(id=1,  vector=[0.85, 0.80, 0.10, 0.10, 0.05, 0.05, 0.15, 0.10],
                payload={"title": "Mad Max: Fury Road",        "genre": "action",  "year": 2015, "rating": 8.1, "director": "Miller"}),
    PointStruct(id=2,  vector=[0.90, 0.75, 0.05, 0.10, 0.05, 0.05, 0.10, 0.10],
                payload={"title": "John Wick",                 "genre": "action",  "year": 2014, "rating": 7.4, "director": "Stahelski"}),
    PointStruct(id=3,  vector=[0.80, 0.70, 0.20, 0.30, 0.05, 0.05, 0.25, 0.20],
                payload={"title": "The Dark Knight",           "genre": "action",  "year": 2008, "rating": 9.0, "director": "Nolan"}),
    PointStruct(id=15, vector=[0.85, 0.80, 0.15, 0.10, 0.10, 0.05, 0.20, 0.15],
                payload={"title": "Avengers: Endgame",         "genre": "action",  "year": 2019, "rating": 8.4, "director": "Russo"}),
    # --- Drama ---
    PointStruct(id=4,  vector=[0.10, 0.10, 0.90, 0.85, 0.10, 0.10, 0.05, 0.05],
                payload={"title": "The Shawshank Redemption",  "genre": "drama",   "year": 1994, "rating": 9.3, "director": "Darabont"}),
    PointStruct(id=5,  vector=[0.10, 0.10, 0.85, 0.80, 0.20, 0.15, 0.05, 0.05],
                payload={"title": "Forrest Gump",              "genre": "drama",   "year": 1994, "rating": 8.8, "director": "Zemeckis"}),
    PointStruct(id=6,  vector=[0.10, 0.15, 0.85, 0.75, 0.05, 0.05, 0.10, 0.05],
                payload={"title": "Parasite",                  "genre": "drama",   "year": 2019, "rating": 8.5, "director": "Bong"}),
    PointStruct(id=14, vector=[0.15, 0.10, 0.75, 0.70, 0.05, 0.05, 0.20, 0.10],
                payload={"title": "Get Out",                   "genre": "drama",   "year": 2017, "rating": 7.7, "director": "Peele"}),
    # --- Comedy ---
    PointStruct(id=7,  vector=[0.10, 0.10, 0.20, 0.15, 0.85, 0.80, 0.05, 0.05],
                payload={"title": "The Grand Budapest Hotel",  "genre": "comedy",  "year": 2014, "rating": 8.1, "director": "Anderson"}),
    PointStruct(id=8,  vector=[0.10, 0.10, 0.15, 0.10, 0.90, 0.85, 0.05, 0.05],
                payload={"title": "Superbad",                  "genre": "comedy",  "year": 2007, "rating": 7.6, "director": "Mottola"}),
    PointStruct(id=9,  vector=[0.15, 0.10, 0.15, 0.10, 0.85, 0.80, 0.05, 0.05],
                payload={"title": "The Hangover",              "genre": "comedy",  "year": 2009, "rating": 7.7, "director": "Phillips"}),
    # --- Sci-Fi ---
    PointStruct(id=10, vector=[0.15, 0.10, 0.30, 0.25, 0.05, 0.05, 0.85, 0.80],
                payload={"title": "Interstellar",              "genre": "sci-fi",  "year": 2014, "rating": 8.6, "director": "Nolan"}),
    PointStruct(id=11, vector=[0.30, 0.25, 0.10, 0.10, 0.05, 0.05, 0.85, 0.80],
                payload={"title": "The Matrix",                "genre": "sci-fi",  "year": 1999, "rating": 8.7, "director": "Wachowski"}),
    PointStruct(id=12, vector=[0.10, 0.15, 0.25, 0.20, 0.05, 0.05, 0.80, 0.85],
                payload={"title": "Blade Runner 2049",         "genre": "sci-fi",  "year": 2017, "rating": 8.0, "director": "Villeneuve"}),
    PointStruct(id=13, vector=[0.25, 0.20, 0.25, 0.20, 0.05, 0.05, 0.85, 0.80],
                payload={"title": "Inception",                 "genre": "sci-fi",  "year": 2010, "rating": 8.8, "director": "Nolan"}),
]
# fmt: on

client.upsert(collection_name=COLLECTION, points=MOVIES)

# Create payload indexes immediately after inserting data.
# This Qdrant Cloud cluster is configured with strict filtering mode,
# which requires indexes to exist before filtered queries can run.
# Indexes also speed up filter evaluation at any scale.
# Index all fields we will filter on.
# This Qdrant Cloud cluster runs in strict mode: every filtered field must be indexed.
for field, schema in [
    ("genre",    PayloadSchemaType.KEYWORD),
    ("title",    PayloadSchemaType.KEYWORD),
    ("director", PayloadSchemaType.KEYWORD),
    ("year",     PayloadSchemaType.INTEGER),
    ("rating",   PayloadSchemaType.FLOAT),
]:
    client.create_payload_index(
        collection_name=COLLECTION,
        field_name=field,
        field_schema=schema,
    )

info = client.get_collection(COLLECTION)
print(f"Inserted {info.points_count} movies into '{COLLECTION}'")
print('Indexed fields:', list(info.payload_schema.keys()))

Inserted 15 movies into 'movies'
Indexed fields: ['genre', 'director', 'title', 'year', 'rating']


## 2. Filtered Search

Filters let you **narrow down search results** — like searching for
"movies similar to Inception" but only among sci-fi films released after 2010.

Qdrant filter conditions:
- `MatchValue` — field equals an exact value
- `MatchAny` — field matches any value in a list
- `Range` — field is within a numeric range

Combine them with `Filter(must=[...], must_not=[...], should=[...])`
— equivalent to SQL `AND` / `NOT` / `OR`.


In [4]:
# ── MatchValue ───────────────────────────────────────────────────────────
# "Find movies similar to The Dark Knight, but ONLY action films."
# The Dark Knight vector has some drama/sci-fi influence;
# the filter ensures only genre="action" results come back.

dark_knight_vector = [0.80, 0.70, 0.20, 0.30, 0.05, 0.05, 0.25, 0.20]

results = client.query_points(
    collection_name=COLLECTION,
    query=dark_knight_vector,
    query_filter=Filter(
        must=[FieldCondition(key="genre", match=MatchValue(value="action"))]
    ),
    limit=5,
)

print('Movies similar to The Dark Knight — action only:')
for p in results.points:
    print(f"  [{p.score:.3f}] {p.payload['title']} ({p.payload['year']})")

Movies similar to The Dark Knight — action only:
  [1.000] The Dark Knight (2008)
  [0.979] Avengers: Endgame (2019)
  [0.971] Mad Max: Fury Road (2015)
  [0.961] John Wick (2014)


In [5]:
# ── Range ────────────────────────────────────────────────────────────────
# "Find sci-fi movies similar to Inception, released in 2010 or later."
# Range(gte=X) means greater-than-or-equal. Also available: gt, lt, lte.

inception_vector = [0.25, 0.20, 0.25, 0.20, 0.05, 0.05, 0.85, 0.80]

results = client.query_points(
    collection_name=COLLECTION,
    query=inception_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key="genre", match=MatchValue(value="sci-fi")),
            FieldCondition(key="year",  range=Range(gte=2010)),
        ]
    ),
    limit=5,
)

print('Sci-fi movies similar to Inception, released 2010 or later:')
for p in results.points:
    print(f"  [{p.score:.3f}] {p.payload['title']} ({p.payload['year']})")

Sci-fi movies similar to Inception, released 2010 or later:
  [1.000] Inception (2010)
  [0.992] Interstellar (2014)
  [0.991] Blade Runner 2049 (2017)


In [6]:
# ── MatchAny + must_not ──────────────────────────────────────────────────
# "Find movies similar to John Wick from action OR sci-fi,
#  but NOT The Dark Knight (we already know that one)."
# MatchAny matches if the field equals any value in the provided list.

john_wick_vector = [0.90, 0.75, 0.05, 0.10, 0.05, 0.05, 0.10, 0.10]

results = client.query_points(
    collection_name=COLLECTION,
    query=john_wick_vector,
    query_filter=Filter(
        must=[FieldCondition(key="genre", match=MatchAny(any=["action", "sci-fi"]))],
        must_not=[FieldCondition(key="title", match=MatchValue(value="The Dark Knight"))],
    ),
    limit=5,
)

print('Action or Sci-Fi similar to John Wick (excluding The Dark Knight):')
for p in results.points:
    print(f"  [{p.score:.3f}] {p.payload['title']} ({p.payload['genre']})")

Action or Sci-Fi similar to John Wick (excluding The Dark Knight):
  [1.000] John Wick (action)
  [0.996] Mad Max: Fury Road (action)
  [0.990] Avengers: Endgame (action)
  [0.436] The Matrix (sci-fi)
  [0.388] Inception (sci-fi)


In [7]:
# ── Rating filter + score_threshold ─────────────────────────────────────
# "Find highly-rated dramas (rating >= 8.5) similar to Forrest Gump,
#  and only return results with cosine similarity >= 0.80."
# score_threshold is applied AFTER the vector search; use it to enforce
# a minimum quality bar on the similarity score itself.

forrest_vector = [0.10, 0.10, 0.85, 0.80, 0.20, 0.15, 0.05, 0.05]

results = client.query_points(
    collection_name=COLLECTION,
    query=forrest_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key="genre",  match=MatchValue(value="drama")),
            FieldCondition(key="rating", range=Range(gte=8.5)),
        ]
    ),
    score_threshold=0.8,
    limit=5,
)

print('Highly-rated dramas (rating >= 8.5, similarity >= 0.80) similar to Forrest Gump:')
for p in results.points:
    print(f"  [{p.score:.3f}] {p.payload['title']} (rating={p.payload['rating']})")

Highly-rated dramas (rating >= 8.5, similarity >= 0.80) similar to Forrest Gump:
  [1.000] Forrest Gump (rating=8.8)
  [0.995] The Shawshank Redemption (rating=9.3)
  [0.986] Parasite (rating=8.5)


## 3. Payload Management

**Payloads are metadata attached to a vector point** — like tags on a photo.
You can add, update, partially delete, or completely replace payload fields
without touching the stored vector.

| Operation | Effect |
|-----------|--------|
| `set_payload` | Add or update specific fields (merges) |
| `delete_payload` | Remove specific field keys |
| `overwrite_payload` | Replace the **entire** payload |

**Payload indexing** speeds up filtered searches — similar to adding a database index.
Keyword-indexed fields are also required for the `facet()` API.

> Indexes for `genre`, `title`, `director`, `year`, and `rating` were already created in
> Section 1 (strict-mode requirement). Here we demonstrate the payload modification
> operations and add a new index for `box_office_m`.


In [8]:
# ── set_payload ──────────────────────────────────────────────────────────
# Add new fields to The Dark Knight (id=3) and Inception (id=13).
# set_payload merges the new fields into the existing payload;
# any fields not mentioned are left untouched.

client.set_payload(
    collection_name=COLLECTION,
    payload={"awards": ["Oscar", "BAFTA"], "box_office_m": 1005},
    points=[3],  # The Dark Knight
)
client.set_payload(
    collection_name=COLLECTION,
    payload={"awards": ["BAFTA"], "box_office_m": 836},
    points=[13],  # Inception
)

points = client.retrieve(collection_name=COLLECTION, ids=[3, 13], with_payload=True)
print('After set_payload:')
for p in points:
    print(f"  id={p.id} | {p.payload['title']} | awards={p.payload.get('awards')} | box_office={p.payload.get('box_office_m')}M")

After set_payload:
  id=3 | The Dark Knight | awards=['Oscar', 'BAFTA'] | box_office=1005M
  id=13 | Inception | awards=['BAFTA'] | box_office=836M


In [9]:
# ── delete_payload ───────────────────────────────────────────────────────
# Remove only the 'box_office_m' key from The Dark Knight (id=3).
# All other fields (title, genre, awards, ...) remain intact.

client.delete_payload(
    collection_name=COLLECTION,
    keys=["box_office_m"],
    points=[3],
)

p = client.retrieve(collection_name=COLLECTION, ids=[3], with_payload=True)[0]
print(f'After delete_payload — id=3 payload: {p.payload}')

After delete_payload — id=3 payload: {'title': 'The Dark Knight', 'genre': 'action', 'year': 2008, 'rating': 9.0, 'director': 'Nolan', 'awards': ['Oscar', 'BAFTA']}


In [10]:
# ── overwrite_payload ────────────────────────────────────────────────────
# Completely replace the payload of Superbad (id=8).
# WARNING: this removes ALL existing fields and sets only what you supply.
# Use set_payload if you want to keep existing fields.

client.overwrite_payload(
    collection_name=COLLECTION,
    payload={"title": "Superbad", "genre": "comedy", "year": 2007,
             "rating": 7.6, "director": "Mottola", "note": "payload fully replaced"},
    points=[8],
)

p = client.retrieve(collection_name=COLLECTION, ids=[8], with_payload=True)[0]
print(f'After overwrite_payload — id=8 payload: {p.payload}')

After overwrite_payload — id=8 payload: {'title': 'Superbad', 'genre': 'comedy', 'year': 2007, 'rating': 7.6, 'director': 'Mottola', 'note': 'payload fully replaced'}


In [11]:
# ── create_payload_index ─────────────────────────────────────────────────
# All the fields we actively use were indexed in Section 1.
# Here we demonstrate adding a NEW index for 'box_office_m' (set by set_payload above).
#
# KEYWORD  — for exact-match filters (MatchValue, MatchAny) and facets
# INTEGER  — for range filters on integer fields
# FLOAT    — for range filters on float fields
#
# Without an index, Qdrant does a full payload scan for each filter.
# With an index, it jumps directly to matching points — much faster at scale.

client.create_payload_index(
    collection_name=COLLECTION,
    field_name="box_office_m",
    field_schema=PayloadSchemaType.FLOAT,
)

info = client.get_collection(COLLECTION)
print('All payload indexes on the collection:')
for field, schema in info.payload_schema.items():
    print(f'  {field}: {schema.data_type}')

All payload indexes on the collection:
  year: integer
  director: keyword
  title: keyword
  genre: keyword
  box_office_m: float
  rating: float


## 4. Scroll / Pagination

`scroll()` lets you **page through all points** in a collection without a query vector.
Think of it like scrolling through a feed — it retrieves everything, page by page.

How the offset cursor works:
1. First call: `offset=None` → returns the first page + a `next_offset` value
2. Pass `next_offset` as the `offset` in the next call to get the following page
3. When `next_offset` is `None`, you've reached the end


In [12]:
# Paginate through all movies, 4 per page.
# scroll() returns a tuple: (list_of_points, next_offset).
# next_offset is None when there are no more results.

PAGE_SIZE = 4
offset = None
page_num = 0
all_retrieved = []

while True:
    page_num += 1
    page_points, next_offset = client.scroll(
        collection_name=COLLECTION,
        limit=PAGE_SIZE,
        offset=offset,
        with_payload=True,
    )

    print(f'--- Page {page_num} (offset={offset}) ---')
    for p in page_points:
        print(f"  id={p.id:3d} | {p.payload['title']} ({p.payload['genre']})")
        all_retrieved.append(p)

    if next_offset is None:
        break
    offset = next_offset

print(f'\nTotal points retrieved across all pages: {len(all_retrieved)}')

--- Page 1 (offset=None) ---
  id=  1 | Mad Max: Fury Road (action)
  id=  2 | John Wick (action)
  id=  3 | The Dark Knight (action)
  id=  4 | The Shawshank Redemption (drama)
--- Page 2 (offset=5) ---
  id=  5 | Forrest Gump (drama)
  id=  6 | Parasite (drama)
  id=  7 | The Grand Budapest Hotel (comedy)
  id=  8 | Superbad (comedy)
--- Page 3 (offset=9) ---
  id=  9 | The Hangover (comedy)
  id= 10 | Interstellar (sci-fi)
  id= 11 | The Matrix (sci-fi)
  id= 12 | Blade Runner 2049 (sci-fi)
--- Page 4 (offset=13) ---
  id= 13 | Inception (sci-fi)
  id= 14 | Get Out (drama)
  id= 15 | Avengers: Endgame (action)

Total points retrieved across all pages: 15


In [13]:
# Scroll also accepts a filter — useful for exporting a filtered subset.
# Example: retrieve all original (non-synthetic) sci-fi movies.

scifi_points, _ = client.scroll(
    collection_name=COLLECTION,
    scroll_filter=Filter(
        must=[FieldCondition(key="genre", match=MatchValue(value="sci-fi"))]
    ),
    limit=50,
    with_payload=True,
)

print('All sci-fi movies (filtered scroll):')
for p in scifi_points:
    print(f"  id={p.id:3d} | {p.payload['title']} ({p.payload['year']})")

All sci-fi movies (filtered scroll):
  id= 10 | Interstellar (2014)
  id= 11 | The Matrix (1999)
  id= 12 | Blade Runner 2049 (2017)
  id= 13 | Inception (2010)


## 5. Recommendation API

Give Qdrant examples of **what you like** (positive IDs) and **what you dislike**
(negative IDs), and it finds more points like your likes — and away from your dislikes.

This is different from a plain vector search:
- **Vector search**: *'find points near this vector'*
- **Recommend**: *'find points similar to these IDs, dissimilar to those IDs'*

Use case: *'I liked John Wick and Mad Max, I disliked Forrest Gump — what else should I watch?'*


In [14]:
# Positive examples: action movies the user liked (ids 1, 2, 3)
# Negative examples: dramas the user disliked (ids 4, 5)
#
# Qdrant averages the positive vectors, subtracts the negative vectors,
# and finds points nearest to the resulting direction.

results = client.query_points(
    collection_name=COLLECTION,
    query=RecommendQuery(
        recommend=RecommendInput(
            positive=[1, 2, 3],   # liked: Mad Max, John Wick, Dark Knight
            negative=[4, 5],      # disliked: Shawshank, Forrest Gump
        )
    ),
    limit=5,
)

print('Recommended (like action, dislike drama):')
for p in results.points:
    print(f"  [score={p.score:.3f}] {p.payload['title']} ({p.payload['genre']}, {p.payload['year']})")

Recommended (like action, dislike drama):
  [score=0.876] Avengers: Endgame (action, 2019)
  [score=0.401] The Matrix (sci-fi, 1999)
  [score=0.297] Inception (sci-fi, 2010)
  [score=0.192] Blade Runner 2049 (sci-fi, 2017)
  [score=0.175] Interstellar (sci-fi, 2014)


In [15]:
# Recommendation + filter: same likes/dislikes,
# but only recommend from movies released after 2010.
# You can combine the recommendation API with any filter.

results = client.query_points(
    collection_name=COLLECTION,
    query=RecommendQuery(
        recommend=RecommendInput(
            positive=[1, 2],   # liked: Mad Max, John Wick
            negative=[7, 8],   # disliked: Grand Budapest, Superbad (comedies)
        )
    ),
    query_filter=Filter(
        must=[FieldCondition(key="year", range=Range(gte=2011))]
    ),
    limit=5,
)

print('Recommended post-2010 movies (like action, dislike comedy):')
for p in results.points:
    print(f"  [score={p.score:.3f}] {p.payload['title']} ({p.payload['genre']}, {p.payload['year']})")

Recommended post-2010 movies (like action, dislike comedy):
  [score=0.854] Avengers: Endgame (action, 2019)
  [score=0.207] Interstellar (sci-fi, 2014)
  [score=0.205] Blade Runner 2049 (sci-fi, 2017)
  [score=0.160] Get Out (drama, 2017)
  [score=0.135] Parasite (drama, 2019)


## 6. Batch Operations

**Processing many operations at once** reduces network round-trips and improves throughput.

- `upload_points` — inserts a large list in configurable chunks
- `query_batch_points` — runs multiple independent searches in a **single request**


In [16]:
# ── upload_points (chunked batch insert) ────────────────────────────────
# upload_points automatically splits the list into chunks of `batch_size`.
# This is the recommended way to insert thousands of points efficiently.

import random

random.seed(42)
genres = ["action", "drama", "comedy", "sci-fi"]
extra_points = []

for i in range(100, 150):  # IDs 100–149 (50 synthetic movies)
    genre_idx = random.randint(0, 3)
    v = [0.05] * 8
    # Make dims for this genre strong
    v[genre_idx * 2]     = 0.7 + random.uniform(-0.1, 0.1)
    v[genre_idx * 2 + 1] = 0.6 + random.uniform(-0.1, 0.1)
    extra_points.append(
        PointStruct(
            id=i,
            vector=v,
            payload={
                "title":    f"Synthetic Movie {i}",
                "genre":    genres[genre_idx],
                "year":     random.randint(1990, 2024),
                "rating":   round(random.uniform(5.0, 9.5), 1),
                "director": "Various",
            },
        )
    )

# batch_size=25: sends 2 requests of 25 points each instead of 50 individual requests
client.upload_points(
    collection_name=COLLECTION,
    points=extra_points,
    batch_size=25,
)

info = client.get_collection(COLLECTION)
print(f"Collection now has {info.points_count} points after batch upload")

Collection now has 65 points after batch upload


In [17]:
# ── query_batch_points ───────────────────────────────────────────────────
# Run multiple independent searches in a single API call.
# Returns a list of result lists — one per QueryRequest.
# Great for dashboards or recommendation widgets that need several results at once.

batch_results = client.query_batch_points(
    collection_name=COLLECTION,
    requests=[
        # Request 1: top action movies similar to an action-heavy vector
        QueryRequest(
            query=[0.88, 0.78, 0.08, 0.08, 0.05, 0.05, 0.12, 0.10],
            filter=Filter(
                must=[FieldCondition(key="genre", match=MatchValue(value="action"))]
            ),
            limit=3,
            with_payload=True,
        ),
        # Request 2: top sci-fi movies similar to a sci-fi vector
        QueryRequest(
            query=[0.20, 0.15, 0.20, 0.15, 0.05, 0.05, 0.82, 0.78],
            filter=Filter(
                must=[FieldCondition(key="genre", match=MatchValue(value="sci-fi"))]
            ),
            limit=3,
            with_payload=True,
        ),
        # Request 3: top comedy movies similar to a comedy vector
        QueryRequest(
            query=[0.10, 0.10, 0.15, 0.10, 0.88, 0.82, 0.05, 0.05],
            filter=Filter(
                must=[FieldCondition(key="genre", match=MatchValue(value="comedy"))]
            ),
            limit=3,
            with_payload=True,
        ),
    ],
)

# Each element in batch_results is a QueryResponse with a .points list
labels = ["Action search", "Sci-Fi search", "Comedy search"]
for label, response in zip(labels, batch_results):
    print(f'\n{label}:')
    for p in response.points:
        print(f"  [{p.score:.3f}] {p.payload['title']} ({p.payload['year']})")


Action search:
  [0.999] John Wick (2014)
  [0.999] Mad Max: Fury Road (2015)
  [0.999] Synthetic Movie 102 (2022)

Sci-Fi search:
  [0.998] Inception (2010)
  [0.994] Blade Runner 2049 (2017)
  [0.992] Interstellar (2014)

Comedy search:
  [1.000] Superbad (2007)
  [0.999] The Hangover (2009)
  [0.998] The Grand Budapest Hotel (2014)


## 7. Grouped Search

**Group results by a payload field** — like asking:
*'Find movies similar to Inception, but give me at most 2 per genre.'*

Without grouping, a single genre might dominate all top-K slots if it scores highest.
Grouping enforces diversity: you always get a fair representation of each group.


In [18]:
# query_points_groups buckets the results by a payload field value.
#
# group_by='genre'  — each bucket is one genre string
# group_size=2      — at most 2 results per genre
# limit=20          — scan the top-20 candidates before bucketing

inception_vector = [0.25, 0.20, 0.25, 0.20, 0.05, 0.05, 0.85, 0.80]

groups_result = client.query_points_groups(
    collection_name=COLLECTION,
    query=inception_vector,
    group_by="genre",
    group_size=2,
    limit=20,
)

print('Movies similar to Inception — at most 2 per genre:')
for group in groups_result.groups:
    print(f'\n  Genre: {group.id}')
    for p in group.hits:
        print(f"    [{p.score:.3f}] {p.payload['title']} ({p.payload['year']})")

Movies similar to Inception — at most 2 per genre:

  Genre: sci-fi
    [1.000] Inception (2010)
    [0.992] Interstellar (2014)

  Genre: action
    [0.564] The Dark Knight (2008)
    [0.478] Avengers: Endgame (2019)

  Genre: drama
    [0.478] Get Out (2017)
    [0.378] Parasite (2019)

  Genre: comedy
    [0.192] The Grand Budapest Hotel (2014)
    [0.187] The Hangover (2009)


In [19]:
# Grouped search also works with filters.
# Example: movies similar to John Wick, grouped by genre, only post-2009.

john_wick_vector = [0.90, 0.75, 0.05, 0.10, 0.05, 0.05, 0.10, 0.10]

groups_result = client.query_points_groups(
    collection_name=COLLECTION,
    query=john_wick_vector,
    group_by="genre",
    group_size=2,
    limit=20,
    query_filter=Filter(
        must=[FieldCondition(key="year", range=Range(gte=2010))]
    ),
)

print('Movies similar to John Wick (post-2009), at most 2 per genre:')
for group in groups_result.groups:
    print(f'\n  Genre: {group.id}')
    for p in group.hits:
        print(f"    [{p.score:.3f}] {p.payload['title']} ({p.payload['year']})")

Movies similar to John Wick (post-2009), at most 2 per genre:

  Genre: action
    [1.000] John Wick (2014)
    [0.998] Synthetic Movie 114 (2010)

  Genre: sci-fi
    [0.388] Inception (2010)
    [0.284] Interstellar (2014)

  Genre: drama
    [0.278] Get Out (2017)
    [0.248] Parasite (2019)

  Genre: comedy
    [0.198] The Grand Budapest Hotel (2014)
    [0.157] Synthetic Movie 105 (2011)


## 8. Count & Facets

- **`count()`** — how many points match a condition?
  Analogous to `SELECT COUNT(*) FROM movies WHERE year > 2015`.

- **`facet()`** — how many points fall into each category of a field?
  Analogous to `SELECT genre, COUNT(*) FROM movies GROUP BY genre`.
  Requires the field to have a keyword index (created in Section 3).


In [20]:
# ── count ───────────────────────────────────────────────────────────────

# Total points in the collection (no filter)
total = client.count(collection_name=COLLECTION)
print(f'Total movies in collection: {total.count}')

# Count per genre
print("\nCount per genre:")
for genre in ["action", "drama", "comedy", "sci-fi"]:
    result = client.count(
        collection_name=COLLECTION,
        count_filter=Filter(
            must=[
                FieldCondition(key="genre", match=MatchValue(value=genre))
            ]
        ),
    )
    print(f'  {genre:8s}: {result.count}')

# Count recent, high-quality movies
result = client.count(
    collection_name=COLLECTION,
    count_filter=Filter(
        must=[
            FieldCondition(key="year",   range=Range(gte=2014)),
            FieldCondition(key="rating", range=Range(gte=8.0)),
        ]
    ),
)
print(f'\nMovies from 2014+ with rating >= 8.0: {result.count}')

Total movies in collection: 65

Count per genre:
  action  : 18
  drama   : 16
  comedy  : 13
  sci-fi  : 18

Movies from 2014+ with rating >= 8.0: 11


In [21]:
# ── facet ───────────────────────────────────────────────────────────────
# facet() counts occurrences of each unique value for a keyword-indexed field.
# It is equivalent to GROUP BY in SQL, without needing a query vector.

response = client.facet(
    collection_name=COLLECTION,
    key="genre",
    limit=10,  # return at most 10 distinct values
)

print('Genre breakdown (all movies):')
for hit in response.hits:
    bar = "█" * min(hit.count, 30)
    print(f'  {hit.value:8s}: {bar} ({hit.count})')

Genre breakdown (all movies):
  action  : ██████████████████ (18)
  sci-fi  : ██████████████████ (18)
  drama   : ████████████████ (16)
  comedy  : █████████████ (13)


In [22]:
# Facet with a filter: genre breakdown only for post-2009 movies.
# This lets you answer 'what is the genre distribution of recent movies?'

response = client.facet(
    collection_name=COLLECTION,
    key="genre",
    facet_filter=Filter(
        must=[FieldCondition(key="year", range=Range(gte=2010))]
    ),
    limit=10,
)

print('Genre breakdown for movies released 2010 and later:')
for hit in response.hits:
    print(f'  {hit.value:8s}: {hit.count}')

Genre breakdown for movies released 2010 and later:
  sci-fi  : 10
  action  : 9
  comedy  : 6
  drama   : 6


## 9. Cleanup

Delete the `movies` collection to free resources on the Qdrant Cloud instance.


In [ ]:
# client.delete_collection(COLLECTION)
# print(f"Collection '{COLLECTION}' deleted.")
# print('Remaining collections:', client.get_collections())